Agent Environment Loop

In [47]:
from typing import Any, Dict, List, Tuple, Optional
from dataclasses import dataclass, field
import abc, time, random, numpy as np
import gymnasium as gym
import multiprocessing
import minigrid

Base Agent Class Interface

In [7]:
class Agent(abc.ABC):
    """Abstract Base Class defining standard RL Agent Interface"""
    
    @abc.abstractmethod
    def observe(self, observation: Any) -> None:
        """Receive and process the current environment state or observation"""
        pass

    @abc.abstractmethod
    def act(self) -> int:
        """Select and return an action based on the agent's current policy"""
        pass

    @abc.abstractmethod
    def update(self, reward: float, terminated: bool, truncated: bool) -> None:
        """Update internal weights, tables or history based on environment feedback"""
        pass

Initialise Random Agent

In [12]:
class RandomAgent(Agent):
    def __init__(self, action_space: gym.spaces.Discrete):
        self.action_space = action_space
    
    # A random agent does not observe neither does it update its internal state. It acts randomly.
    def observe(self, observation: Any) -> None:
        pass

    def update(self, reward: float, terminated: bool, truncated: bool) -> None:
        pass

    def act(self) -> int:
        return int(self.action_space.sample())

Initializing Cartpole Heuristics Agent

In [13]:
class CartpoleHeuristicsAgent(Agent):
    """Move the cart in the same direction the pole is tilting to balance it... as simple as that"""
    def __init__(self) -> None:
        self.pole_angle = 0.0
    
    def observe(self, observation: Any) -> None:
        # observation layout: [cart_pos, cart_vel, pole_angle, pole_vel]
        self.pole_angle = float(observation[2])
    
    # This agent uses fixed logic and does not learn
    def update(self, reward: float, terminated: bool, truncated: bool) -> None:
        pass

    def act(self) -> int:
        return 1 if self.pole_angle > 0 else 0

Initializing MiniGrid Heuristics Agent

In [15]:
class MiniGridHeuristicsAgent(Agent):
    """Rule: Move forward, if block by a wall, rotate right"""
    def __init__(self) -> None:
        self.is_front_clear: bool = True

    def observe(self, observation: Any) -> None:
        # observation['image'] - 7 x 7 image - space directly infront is at index (5, 3)
        grid = observation['image']
        front_cell = grid[5, 3]
        self.is_front_clear = (front_cell[0] != 2)
    
    def update(self, reward: float, terminated: bool, truncated: bool) -> None:
        pass

    def act(self) -> int:
        # 2 - forward, 1 - right, 0 - left
        return 2 if self.is_front_clear else 1

Initializing TableBased Agent

In [40]:
class TableBasedAgent(Agent):
    """A simple tabular Q-learning agent"""
    def __init__(self, action_space: gym.spaces.Discrete, is_vector_env: bool = True):
        self.action_space = action_space
        self.is_vector_env = is_vector_env
        self.q_table: Dict[Tuple[int, ...], Dict[int, float]] = {}
        self.current_state_key: Optional[Tuple[int, ...]] = None
        self.last_state_key: Optional[Tuple[int, ...]] = None
        self.last_action: Optional[int] = None

        # Hyperparameters
        self.alpha = 0.1 # the learning rate
        self.gamma = 0.99 # the discount factor - does it prioritise long-term rewards over immediate satisfaction
        self.epsilon = 0.1 # the exploration rate - vs exploitation
    
    def _discretize(self, observation: Any) -> Tuple[int, ...]:
        """Maps complex observation layouts to discrete hashable tuples"""
        if self.is_vector_env: # for cartpole
            return tuple(np.round(observation, decimals=1).tolist())
        else: # for minigrid
            pos_encoding = tuple(observation['image'][:, :, 0].flatten().tolist())
            return (observation['direction'], *pos_encoding)
    
    def observe(self, observation: Any) -> None:
        state_key = self._discretize(observation)
        self.last_state_key = self.current_state_key
        self.current_state_key = state_key
        if state_key not in self.q_table:
            self.q_table[state_key] = {a: 0.0 for a in range(self.action_space.n)}
    
    def act(self) -> int:
        state_values = self.q_table[self.current_state_key]
        if random.random() < self.epsilon:
            self.last_action = int(self.action_space.sample())
        else:
            self.last_action = max(state_values, key=state_values.get)
        return self.last_action
    
    def update(self, reward: float, terminated: bool, truncated: bool) -> None:
        if self.last_state_key is None and self.last_action is None:
            return
        current_q = self.q_table[self.last_state_key][self.last_action]
        max_future_q = 0.0 if (terminated or truncated) else max(self.q_table[self.current_state_key].values())
        self.q_table[self.last_state_key][self.last_action] = current_q + self.alpha * (
            reward + self.gamma * max_future_q - current_q
        )

Initiatlize DelayedMalfunctioning Agent

In [42]:
class DelayedMalfunctioningAgent(Agent):
    """Slow agent to test safety budgets"""
    def observe(self, observation: Any) -> None:
        pass
    def update(self, reward: float, terminated: bool, truncated: bool) -> None:
        pass
    def act(self) -> int:
        time.sleep(0.05) # forcing a 50ms delay
        return 0

Logging Infra and Budget Runner

In [46]:
@dataclass
class EpisodicLog:
    agent_name: str
    environment_name: str
    episodic_return: float = 0.0
    episodic_length: int = 0
    action_distribution: Dict[int, int] = field(default_factory=dict)
    failure_states: List[int] = field(default_factory=list)

In [48]:
def _worker_act(agent: Agent, queue: multiprocessing.Queue):
    try:
        action = agent.act()
        queue.put(("True", action))
    except Exception as e:
        queue.put(("False", str(e)))

In [50]:
def get_action_with_budget(agent: Agent, time_budget: float) -> Tuple[Optional[int], Optional[str]]:
    queue = multiprocessing.get_context("spawn").Queue()
    process = multiprocessing.get_context("spawn").Process(target=_worker_act, args=(agent, queue))
    process.start()
    process.join(timeout=time_budget)

    if process.is_alive():
        process.terminate()
        process.join()
        return None, f"Timeout budget exceeded (> {time_budget*1000:.1f}ms)"

    if not queue.empty():
        success, val = queue.get()
        if success:
            return val, None
        return None, f"Execution Error: {val}"
    return None, "Process died unexpectedly"